---
title: "Salary Linear Regression Model"
format:
  html:
    code-fold: false
    code-overflow: wrap
    code-tools: false
    echo: false
    warning: false
    message: false
---

In [1]:
# Import Necessary Libraries
import pandas as pd
import seaborn as sns
from sklearn import preprocessing
from sklearn.cluster import KMeans
from sklearn import metrics
from sklearn.linear_model import LinearRegression
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import plotly.express as px
import os 
import plotly.figure_factory as ff
import plotly.io as pio

pio.renderers.default = 'jupyterlab+plotly_mimetype' 

In [2]:
# Load in Dataset
jobdata = pd.read_csv("data/jobs_in_data_2024.csv")

# Introduction

## Exploratory Data Analysis

In [3]:
# Display First 10 Rows of Dataset
jobdata.head(10)

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,work_setting,company_location,company_size,job_category
0,2024,Entry-level,Freelance,Applied Data Scientist,30000,USD,30000,United Kingdom,Remote,United Kingdom,M,Data Science and Research
1,2024,Executive,Full-time,Business Intelligence,230000,USD,230000,United States,In-person,United States,M,BI and Visualization
2,2024,Executive,Full-time,Business Intelligence,176900,USD,176900,United States,In-person,United States,M,BI and Visualization
3,2024,Senior,Full-time,Data Architect,171210,USD,171210,Canada,In-person,Canada,M,Data Architecture and Modeling
4,2024,Senior,Full-time,Data Architect,92190,USD,92190,Canada,In-person,Canada,M,Data Architecture and Modeling
5,2024,Mid-level,Full-time,Data Science,46203,GBP,57753,United Kingdom,In-person,United Kingdom,M,Data Science and Research
6,2024,Mid-level,Full-time,Data Science,38280,GBP,47850,United Kingdom,In-person,United Kingdom,M,Data Science and Research
7,2024,Entry-level,Full-time,Insight Analyst,50000,USD,50000,United States,Remote,United States,M,Data Analysis
8,2024,Entry-level,Full-time,Insight Analyst,40000,USD,40000,United States,Remote,United States,M,Data Analysis
9,2024,Senior,Full-time,Data Engineer,276000,USD,276000,United States,In-person,United States,M,Data Engineering


In [4]:
# Check for Missing Values
jobdata.isnull().sum()

work_year             0
experience_level      0
employment_type       0
job_title             0
salary                0
salary_currency       0
salary_in_usd         0
employee_residence    0
work_setting          0
company_location      0
company_size          0
job_category          0
dtype: int64

In [5]:
jobdata['salary_in_usd'].describe()

count     14199.00000
mean     149472.04944
std       64379.26016
min       15000.00000
25%      104000.00000
50%      142000.00000
75%      185900.00000
max      450000.00000
Name: salary_in_usd, dtype: float64

In [6]:
import plotly.express as px
import plotly.io as pio
import pandas as pd # Assuming jobdata is a pandas DataFrame

# --- 1. PLOTLY RENDERER SETUP (CRITICAL FOR QUARTO) ---
# This ensures the output is a self-contained HTML/MIME type 
# that Quarto can read and embed during the render process.
pio.renderers.default = 'plotly_mimetype+jupyterlab' 
# 'jupyterlab' is a reliable fallback for VS Code/Jupyter environments.

# --- 2. HISTOGRAM CODE ---

# Assuming 'jobdata' is the pandas DataFrame and is already loaded

# Create the histogram
fig = px.histogram(
    jobdata,
    x='salary_in_usd',
    nbins=50,  # Sets the number of bins
    title='Distribution of Salary in USD'
)

# Update axis titles and layout
fig.update_layout(
    xaxis_title='Salary in USD',
    yaxis_title='Count',
    margin=dict(l=40, r=40, t=50, b=40)
)

fig.show() # The fig.show() command outputs the configured MIME type

In [14]:
# Visualize Salary Distribution

# Prepare the data for Plotly's distribution plot
data_to_plot = [jobdata['salary_in_usd'].dropna().tolist()]

# Create the distribution plot (histogram + KDE line)
fig_dist = ff.create_distplot(
    data_to_plot,
    ['Salary in USD'], 
    show_hist=True,    # Show the histogram bars
    show_curve=True,   # Show the KDE line ("trend line")
    show_rug=False     
)

# Update layout for titles and axis labels
fig_dist.update_layout(
    title_text='Distribution of Salary in USD',
    xaxis_title='Salary in USD',
    yaxis_title='Density', # Y-axis must be 'Density' with KDE
    margin=dict(l=40, r=40, t=50, b=40),
    height=600 
)

# --- Optional: Save plot to the 'plots' directory ---
os.makedirs("plots", exist_ok=True)
fig_dist.write_html("plots/salary_distribution_kde.html", include_plotlyjs="cdn")
# --- END OPTIONAL LINES ---

fig_dist.show()

In [8]:

avg_salary_df = jobdata.groupby('experience_level', as_index=False)['salary_in_usd'].mean().sort_values(by='salary_in_usd', ascending=True)

fig = px.bar(
    avg_salary_df,
    x='experience_level',
    y='salary_in_usd',
    title='Average Salary by Experience Level',
)

fig.update_layout(
    xaxis_title='Experience Level',
    yaxis_title='Average Salary in USD',
    hoverlabel=dict(
        bgcolor="white",
        font_size=12,
        font_family="sans-serif"
    )
)

fig.show()

In [15]:
# Visualize Average Salary by Experience Level

# 1. Aggregate and Sort the Data (Replicates the 'order' calculation)
avg_salary_df = jobdata.groupby('experience_level', as_index=False)['salary_in_usd'].mean().sort_values(by='salary_in_usd', ascending=True)

# 2. Create the Plotly Bar Chart
fig_bar = px.bar(
    avg_salary_df,
    x='experience_level',
    y='salary_in_usd',
    title='Average Salary by Experience Level',
)

# 3. Update Layout for final titles and formatting
fig_bar.update_layout(
    xaxis_title='Experience Level',
    yaxis_title='Average Salary in USD'
)

# --- Optional: Save plot to the 'plots' directory ---
os.makedirs("plots", exist_ok=True)
fig_bar.write_html("plots/average_salary_by_experience.html", include_plotlyjs="cdn")
# --- END OPTIONAL LINES ---

fig_bar.show()

In [16]:
import plotly.express as px

# Assuming 'jobdata' is the pandas DataFrame and is already loaded

# Create the Plotly Box Plot
fig = px.box(
    jobdata,
    x='job_category',
    y='salary_in_usd',
    title='Salary Distribution by Job Category'
)

# Update Layout for cleaner titles and to replicate the rotation
fig.update_layout(
    xaxis_title='Job Category',
    yaxis_title='Salary in USD',
    # Replicate the plt.xticks(rotation=45) for better label readability
    xaxis={'tickangle': 45},
    margin=dict(l=40, r=40, t=50, b=40)
)

fig.show()

In [17]:
# Visualize Salary Distribution by Job Category

# Create the Plotly Box Plot
fig_box = px.box(
    jobdata,
    x='job_category',
    y='salary_in_usd',
    title='Salary Distribution by Job Category'
)

# Update Layout for cleaner titles and to replicate the rotation
fig_box.update_layout(
    xaxis_title='Job Category',
    yaxis_title='Salary in USD',
    # Replicate the plt.xticks(rotation=45) for better label readability
    xaxis={'tickangle': 45},
    margin=dict(l=40, r=40, t=50, b=40)
)

# --- Optional: Save plot to the 'plots' directory ---
os.makedirs("plots", exist_ok=True)
fig_box.write_html("plots/salary_by_job_category_boxplot.html", include_plotlyjs="cdn")
# --- END OPTIONAL LINES ---

fig_box.show()

## Prepare Data for Linear Regression Model

In [10]:
# Select Relevant Columns for Analysis
jobdata2 = jobdata[['salary_in_usd', 'experience_level', 'employment_type', 'job_category', 'work_setting']]

In [11]:
# Dummify Categorical Variables
jobdata_regression = pd.get_dummies(jobdata2, drop_first=True, columns = ['experience_level', 'employment_type', 'job_category', 'work_setting'])

In [12]:
# Display First 5 Rows of Dummified Dataset
jobdata_regression.head()

,salary_in_usd,experience_level_Executive,experience_level_Mid-level,experience_level_Senior,employment_type_Freelance,employment_type_Full-time,employment_type_Part-time,job_category_Cloud and Database,job_category_Data Analysis,job_category_Data Architecture and Modeling,job_category_Data Engineering,job_category_Data Management and Strategy,job_category_Data Quality and Operations,job_category_Data Science and Research,job_category_Leadership and Management,job_category_Machine Learning and AI,work_setting_In-person,work_setting_Remote
0,30000,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,True
1,230000,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False
2,176900,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False
3,171210,False,False,True,False,True,False,False,False,True,False,False,False,False,False,False,True,False
4,92190,False,False,True,False,True,False,False,False,True,False,False,False,False,False,False,True,False


In [13]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

NameError: name 'X' is not defined

In [ ]:
# Prepare Data for Linear Regression
X = jobdata_regression[['experience_level_Executive',
       'experience_level_Mid-level', 'experience_level_Senior',
       'employment_type_Freelance', 'employment_type_Full-time',
       'employment_type_Part-time', 'job_category_Cloud and Database',
       'job_category_Data Analysis',
       'job_category_Data Architecture and Modeling',
       'job_category_Data Engineering',
       'job_category_Data Management and Strategy',
       'job_category_Data Quality and Operations',
       'job_category_Data Science and Research',
       'job_category_Leadership and Management',
       'job_category_Machine Learning and AI', 'work_setting_In-person', 'work_setting_Remote']]
y = jobdata_regression['salary_in_usd']


In [ ]:
# Train Linear Regression Model
regressor = LinearRegression()
regressor.fit(X_train,y_train)

In [ ]:
# Get Intercept
regressor.intercept_

In [ ]:
# Get Coefficients with Proper Formatting
jobdata_coef = pd.DataFrame(regressor.coef_, X.columns, columns=['Coefficient'])
jobdata_coef['Coefficient'] = jobdata_coef['Coefficient'].map(lambda x: f"-${abs(x):,.2f}" if x < 0 else f"${x:,.2f}")

In [ ]:
# Display Coefficients DataFrame
jobdata_coef

## Model Evaluation

In [ ]:
# Make Predictions on Test Set
y_pred = regressor.predict(X_test)

In [ ]:
# Evaluate Model Performance with RMSE
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_test, y_pred)
mse

In [ ]:
# Calculate RMSE
import numpy as np
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
rmse

In [ ]:
# Calculate R-squared
from sklearn.metrics import r2_score
r2 = r2_score(y_test, y_pred)
r2

In [ ]:
# Print Coefficients and Intercept
print(f"Intercept: {regressor.intercept_}")